# Comprehensive Evaluation and Benchmarking Guide

This notebook provides a comprehensive guide to evaluation and benchmarking for LLaMA-Factory models, covering:

1. **Automatic Evaluation**: Metrics and automated assessment
2. **Human Evaluation**: Human-in-the-loop evaluation methods
3. **Benchmarking**: Performance comparison across methods
4. **Quality Assessment**: Model quality and capability testing
5. **A/B Testing**: Comparative evaluation frameworks
6. **Statistical Analysis**: Significance testing and analysis

## Table of Contents

- [Setup and Installation](#setup-and-installation)
- [Automatic Evaluation](#automatic-evaluation)
- [Human Evaluation](#human-evaluation)
- [Benchmarking](#benchmarking)
- [Quality Assessment](#quality-assessment)
- [A/B Testing](#ab-testing)
- [Statistical Analysis](#statistical-analysis)
- [Best Practices](#best-practices)


## Setup and Installation

First, let's install the required dependencies for evaluation and benchmarking.


In [ ]:
# Install evaluation dependencies
%pip install -r requirements.txt
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%pip install transformers[torch] datasets evaluate rouge-score bert-score
%pip install matplotlib seaborn plotly pandas numpy scipy scikit-learn
%pip install llamafactory peft accelerate

# Import required libraries
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from llamafactory import ChatModel
import evaluate
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Any
import json
import os
from datetime import datetime

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")


## Automatic Evaluation

Let's implement comprehensive automatic evaluation metrics and methods.


In [ ]:
# Comprehensive Evaluation Metrics Implementation
class ComprehensiveEvaluator:
    """Comprehensive evaluation suite for language models"""

    def __init__(self):
        # Load evaluation metrics
        self.bleu = evaluate.load("bleu")
        self.rouge = evaluate.load("rouge")
        self.bertscore = evaluate.load("bertscore")

    def evaluate_responses(self, predictions, references, input_texts=None):
        """Evaluate model responses using multiple metrics"""

        results = {
            "bleu": {},
            "rouge": {},
            "bertscore": {},
            "length_stats": {},
            "diversity": {},
            "overall_score": 0.0
        }

        # BLEU Score
        try:
            bleu_results = self.bleu.compute(
                predictions=predictions,
                references=[[ref] for ref in references]
            )
            results["bleu"] = bleu_results
        except Exception as e:
            print(f"BLEU evaluation failed: {e}")
            results["bleu"] = {"bleu": 0.0}

        # ROUGE Scores
        try:
            rouge_results = self.rouge.compute(
                predictions=predictions,
                references=references
            )
            results["rouge"] = rouge_results
        except Exception as e:
            print(f"ROUGE evaluation failed: {e}")
            results["rouge"] = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}

        # BERTScore
        try:
            bertscore_results = self.bertscore.compute(
                predictions=predictions,
                references=references,
                lang="en"
            )
            results["bertscore"] = {
                "precision": np.mean(bertscore_results["precision"]),
                "recall": np.mean(bertscore_results["recall"]),
                "f1": np.mean(bertscore_results["f1"])
            }
        except Exception as e:
            print(f"BERTScore evaluation failed: {e}")
            results["bertscore"] = {"precision": 0.0, "recall": 0.0, "f1": 0.0}

        # Length Statistics
        pred_lengths = [len(pred.split()) for pred in predictions]
        ref_lengths = [len(ref.split()) for ref in references]

        results["length_stats"] = {
            "pred_avg_length": np.mean(pred_lengths),
            "ref_avg_length": np.mean(ref_lengths),
            "pred_length_std": np.std(pred_lengths),
            "ref_length_std": np.std(ref_lengths),
            "length_ratio": np.mean(pred_lengths) / np.mean(ref_lengths)
        }

        # Diversity (unique n-grams)
        results["diversity"] = self.calculate_diversity(predictions)

        # Overall score (weighted combination)
        results["overall_score"] = self.calculate_overall_score(results)

        return results

    def calculate_diversity(self, texts, n=2):
        """Calculate diversity using unique n-grams"""
        all_ngrams = set()

        for text in texts:
            words = text.lower().split()
            for i in range(len(words) - n + 1):
                ngram = " ".join(words[i:i+n])
                all_ngrams.add(ngram)

        return {
            "unique_bigrams": len(all_ngrams),
            "diversity_score": len(all_ngrams) / max(1, len(texts))
        }

    def calculate_overall_score(self, results):
        """Calculate overall evaluation score"""
        bleu_score = results["bleu"].get("bleu", 0)
        rouge1_score = results["rouge"].get("rouge1", 0)
        bert_f1 = results["bertscore"].get("f1", 0)

        # Weighted combination
        overall_score = (bleu_score * 0.3 + rouge1_score * 0.4 + bert_f1 * 0.3)

        return overall_score

    def generate_evaluation_report(self, results, model_name="Model"):
        """Generate comprehensive evaluation report"""

        report = f"""
# Evaluation Report: {model_name}
Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Overall Score
- **Overall Score**: {results['overall_score']:.3f}

## BLEU Scores
- **BLEU**: {results['bleu'].get('bleu', 0):.3f}
"""

        if 'precisions' in results['bleu']:
            report += f"- **BLEU-1**: {results['bleu']['precisions'][0]:.3f}\n"
            report += f"- **BLEU-2**: {results['bleu']['precisions'][1]:.3f}\n"
            report += f"- **BLEU-3**: {results['bleu']['precisions'][2]:.3f}\n"
            report += f"- **BLEU-4**: {results['bleu']['precisions'][3]:.3f}\n"

        report += f"""
## ROUGE Scores
- **ROUGE-1**: {results['rouge'].get('rouge1', 0):.3f}
- **ROUGE-2**: {results['rouge'].get('rouge2', 0):.3f}
- **ROUGE-L**: {results['rouge'].get('rougeL', 0):.3f}

## BERTScore
- **Precision**: {results['bertscore'].get('precision', 0):.3f}
- **Recall**: {results['bertscore'].get('recall', 0):.3f}
- **F1**: {results['bertscore'].get('f1', 0):.3f}

## Length Statistics
- **Average Prediction Length**: {results['length_stats']['pred_avg_length']:.1f} words
- **Average Reference Length**: {results['length_stats']['ref_avg_length']:.1f} words
- **Length Ratio**: {results['length_stats']['length_ratio']:.2f}

## Diversity
- **Unique Bigrams**: {results['diversity']['unique_bigrams']}
- **Diversity Score**: {results['diversity']['diversity_score']:.2f}
"""

        return report

# Initialize evaluator
evaluator = ComprehensiveEvaluator()
